# Fake News Detection, TF-IDF Baseline

A binary classifier on the real LIAR dataset (Wang, 2017): 12.8K short political statements from PolitiFact, each with a fact-checker label. I download the original zip from UCSB and use the official train (10,269 rows) and test (1,283 rows) files. LIAR's six labels are collapsed to binary: pants-fire, false and barely-true count as "fake"; half-true, mostly-true and true count as "real".

The honest version of this problem is harder than the binary framing makes it look. Real-world "fake news" is mostly *partially* true, framed misleadingly, or just emphasizes some facts over others. A binary label flattens all of that, which is worth remembering before drawing any product conclusions from these results.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(0)
DATA_DIR = Path("../../data/fake_news")


In [2]:
import urllib.request, zipfile
DATA_DIR = Path("../../data/liar")
ZIP = DATA_DIR / "liar_dataset.zip"
if not (DATA_DIR / "train.tsv").exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://www.cs.ucsb.edu/~william/data/liar_dataset.zip", ZIP)
    zipfile.ZipFile(ZIP).extractall(DATA_DIR)

cols = ["id", "label6", "text", "subject", "speaker", "job", "state", "party",
        "barely_true_c", "false_c", "half_true_c", "mostly_true_c", "pants_fire_c", "context"]
def load(split):
    d = pd.read_csv(DATA_DIR / f"{split}.tsv", sep="\t", header=None, names=cols, quoting=3)
    d["label"] = d["label6"].isin({"pants-fire", "false", "barely-true"}).astype(int)  # 1 = fake
    return d
train_df, test_df = load("train"), load("test")
print(train_df.shape, test_df.shape, "  fake rate (train):", train_df.label.mean().round(3))
train_df[["label6", "text"]].head()

(10269, 15) (1283, 15)   fake rate (train): 0.438


,label6,text
0,false,Says the Annies List political group supports ...
1,half-true,When did the decline of coal start? It started...
2,mostly-true,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,Health care reform legislation is likely to ma...
4,half-true,The economic turnaround started at the end of ...


## TF-IDF + LogReg, with a Naive Bayes comparison

In [3]:
X_train, y_train = train_df["text"].values, train_df["label"].values
X_test, y_test = test_df["text"].values, test_df["label"].values
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

results = {}
for name, clf in [("LogReg", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")),
                  ("MultinomialNB", MultinomialNB())]:
    clf.fit(Xtr, y_train)
    proba = clf.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {
        "auc": round(roc_auc_score(y_test, proba), 3),
    }
    print(f"--- {name} ---")
    print(classification_report(y_test, pred, target_names=["real", "fake"], digits=3))

results

--- LogReg ---
              precision    recall  f1-score   support

        real      0.678     0.659     0.668       727
        fake      0.569     0.590     0.580       556

    accuracy                          0.629      1283
   macro avg      0.623     0.624     0.624      1283
weighted avg      0.631     0.629     0.630      1283

--- MultinomialNB ---
              precision    recall  f1-score   support

        real      0.622     0.840     0.715       727
        fake      0.613     0.331     0.430       556

    accuracy                          0.620      1283
   macro avg      0.617     0.586     0.572      1283
weighted avg      0.618     0.620     0.591      1283



{'LogReg': {'auc': 0.675}, 'MultinomialNB': {'auc': 0.668}}

## Notes / honesty section

On the official LIAR test file (1,283 statements), TF-IDF + class-balanced logistic regression gets 62.9% accuracy, 0.624 macro-F1 and ROC-AUC 0.675. Multinomial Naive Bayes lands at 62.0% accuracy but only catches a third of the fake statements (recall 0.33), so its macro-F1 drops to 0.572. For scale, always guessing "real" would get 56.7% on this test set.

That is in line with what text-only models usually get on binary LIAR (low 60s). Real and fake statements share almost all of their vocabulary, so a bag-of-words model can only pick up weak signals like topic and phrasing. The six-way version is much harder still.

For a serious version of this:
- Use LIAR's six-way labels plus the speaker metadata. The speaker's past "credit history" counts are known to be the strongest single feature.
- Add source/URL features when the data has them. They're often more predictive than the text.
- Don't deploy a binary classifier as the final arbiter. It's a triage tool at best.